---

<div align="center">

# **ANÁLISIS DE SERIES DE TIEMPO**

<div align="center">
  <img src="img/logo_uptc2.jpg" width="120">
</div>

**Profesor:** Duván Cataño  

**Curso:** Estadística para Analítica de Datos 

**Universidad Pedagógica y Tecnológica de Colombia**
</div>

---


<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 


## **Introducción**

Las series de tiempo corresponden a conjuntos de observaciones registradas de manera secuencial a lo largo del tiempo, donde el orden cronológico de los datos es fundamental para su análisis. Debido a que las observaciones suelen presentar dependencia temporal, es posible identificar patrones como tendencia, estacionalidad, ciclos y variaciones aleatorias, los cuales permiten comprender la evolución de un fenómeno y realizar pronósticos sobre su comportamiento futuro. En esta unidad se estudiarán los principales conceptos y técnicas para el análisis exploratorio, la descomposición y la modelación de series de tiempo, utilizando tanto enfoques estadísticos clásicos implementados en Statsmodels (como modelos ARIMA y SARIMA) como herramientas de Scikit-learn (sklearn) para la construcción de modelos de aprendizaje automático orientados al pronóstico, proporcionando una base sólida para resolver problemas reales de Ciencia de Datos mediante metodologías estadísticas y de Machine Learning.

En este notebook nos enfocamos a la implementación en ```scikit-learn```. Para investigar la metodología desarrollada en ```statsmodels``` la cual puede ser consultadas en el siguiente texto : 📕 <a href="https://link.springer.com/book/10.1007/978-3-031-13584-2"> Time Series Python</a>

</div>

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

## 1.Descomposición Aditiva y Multiplicativa

### Concepto
El análisis clásico de series de tiempo asume que una observación $Y_t$ en el instante $t$ se compone de cuatro elementos fundamentales:
* **Tendencia ($T_t$):** La dirección general de largo plazo de la serie (creciente, decreciente o constante).
* **Estacionalidad ($S_t$):** Patrones repetitivos a intervalos regulares de tiempo (diario, mensual, trimestral).
* **Ciclos ($C_t$):** Fluctuaciones de más largo plazo sin una periodicidad fija (asociadas a ciclos económicos). En la práctica suele agruparse junto con la tendencia ($T_t \cdot C_t$).
* **Residuo o Irregularidad ($I_t$ o $E_t$):** Ruido aleatorio no predecible ni atribuible a las componentes estructurales.

Dependiendo de cómo interactúan la amplitud de las variaciones estacionales con el nivel de la tendencia, se definen dos esquemas principales:

1. **Modelo Aditivo:** Se utiliza cuando las fluctuaciones estacionales permanecen de magnitud relativamente constante a lo largo del tiempo, independientemente del nivel general de la serie.

2. **Modelo Multiplicativo:** Se utiliza cuando la magnitud de la variación estacional es proporcional al nivel general de la serie (las oscilaciones estacionales crecen o disminuyen conforme la tendencia sube o baja).

### Fórmulas Matemáticas

#### Modelo Aditivo
$$Y_t = T_t + S_t + C_t + I_t$$

Si agrupamos la tendencia y el ciclo en $T_t$:
$$Y_t = T_t + S_t + I_t$$

#### Modelo Multiplicativo
$$Y_t = T_t.S_t.C_t.I_t$$

O agrupando tendencia y ciclo:
$$Y_t = T_t.S_t.I_t$$

*Transformación Logarítmica:* Un modelo multiplicativo puede linealizarse convirtiéndose en aditivo mediante la aplicación de logaritmos:
$$\ln(Y_t) = \ln(T_t) + \ln(S_t) + \ln(I_t)$$

</div>

---

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose

# 1. Simulación de Serie de Tiempo Multiplicativa
np.random.seed(42)
time = pd.date_range(start="2020-01-01", periods=120, freq="M")
trend = np.linspace(10, 50, 120)
seasonal = 1 + 0.3 * np.sin(2 * np.pi * np.arange(120) / 12)
noise = np.random.normal(loc=1, scale=0.05, size=120)

values_mult = trend * seasonal * noise
df = pd.DataFrame({"Fecha": time, "Valor": values_mult}).set_index("Fecha")

# 2. Descomposición con Statsmodels
decomp = seasonal_decompose(df["Valor"], model="multiplicative", period=12)

# 3. Gráficos Interactivos con Plotly Graph Objects (go)
fig = make_subplots(
    rows=4, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Serie Observada ($Y_t$)", "Tendencia ($T_t$)", "Estacionalidad ($S_t$)", "Residuo ($I_t$)")
)

fig.add_trace(go.Scatter(x=df.index, y=decomp.observed, mode="lines", name="Observado", line=dict(color="#1f77b4")), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomp.trend, mode="lines", name="Tendencia", line=dict(color="#ff7f0e")), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomp.seasonal, mode="lines", name="Estacionalidad", line=dict(color="#2ca02c")), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomp.resid, mode="markers", name="Residuo", marker=dict(color="#d62728", size=4)), row=4, col=1)

fig.update_layout(
    height=800, 
    title_text="<b>Descomposición Multiplicativa de Serie de Tiempo</b>",
    template="plotly_white",
    showlegend=False
)
fig.show()

/var/folders/_6/0ld42hn97t9_vc9j7kzt2xg80000gn/T/ipykernel_12688/3638652409.py:10: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  time = pd.date_range(start="2020-01-01", periods=120, freq="M")


<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

---

## 2. Media Móvil (Moving Average)

### Concepto
El método de **Media Móvil** es una técnica de suavizado utilizada para reducir el ruido de alta frecuencia de una serie de tiempo y resaltar tendencias subyacentes. Funciona calculando el promedio de un subconjunto de puntos dentro de una ventana deslizante de tamaño determinado $k$.

* **Media Móvil Simple (SMA):** Promedio de las $k$ observaciones pasadas.
* **Media Móvil Centrada (CMA):** Utiliza observaciones pasadas y futuras de forma simétrica; ideal para la estimación de la tendencia en la descomposición de series.

### Fórmulas Matemáticas

#### Media Móvil Simple Unilateral (de tamaño $k$)
$$MA_t = \frac{1}{k} \sum_{i=0}^{k-1} Y_{t-i} = \frac{Y_t + Y_{t-1} + \dots + Y_{t-k+1}}{k}$$

#### Media Móvil Centrada (para $k$ impar, ej. $k = 2m + 1$)
$$CMA_t = \frac{1}{2m+1} \sum_{j=-m}^{m} Y_{t+j}$$

#### Media Móvil Ponderada (WMA)
$$WMA_t = \sum_{i=0}^{k-1} w_i Y_{t-i} \quad 	\text{donde} \quad \sum_{i=0}^{k-1} w_i = 1$$

---

</div>

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Simulación de datos
np.random.seed(10)
dates = pd.date_range("2023-01-01", periods=100, freq="D")
raw_data = 50 + np.cumsum(np.random.normal(0, 1, 100))
df_ma = pd.DataFrame({"Fecha": dates, "Precio": raw_data})

# Cálculo de Medias Móviles con Pandas
df_ma["MA_7"] = df_ma["Precio"].rolling(window=7).mean()
df_ma["MA_30"] = df_ma["Precio"].rolling(window=30).mean()

# Gráfico con Plotly Express / Graph Objects
fig = go.Figure()

fig.add_trace(go.Scatter(x=df_ma["Fecha"], y=df_ma["Precio"], mode="lines", name="Precio Real", line=dict(color="grey", width=1.5)))
fig.add_trace(go.Scatter(x=df_ma["Fecha"], y=df_ma["MA_7"], mode="lines", name="Media Móvil 7 días", line=dict(color="#00CC96", width=2)))
fig.add_trace(go.Scatter(x=df_ma["Fecha"], y=df_ma["MA_30"], mode="lines", name="Media Móvil 30 días", line=dict(color="#AB63FA", width=2.5)))

fig.update_layout(
    title="<b>Suavizado por Media Móvil (7 vs 30 días)</b>",
    xaxis_title="Fecha",
    yaxis_title="Valor",
    template="plotly_white"
)
fig.show()

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 
---

## 3. Suavización Exponencial Simple (SES)

### Concepto
La **Suavización Exponencial Simple (SES)** es adecuada para pronosticar series temporales que no presentan una tendencia marcada ni un patrón estacional predecible. A diferencia de la media móvil simple (donde todas las observaciones históricas dentro de la ventana tienen la misma ponderación), SES asigna ponderaciones exponencialmente decrecientes a las observaciones a medida que son más antiguas.

El parámetro de suavizado $\alpha \in [0, 1]$ controla la tasa de decaimiento:
- $\alpha \approx 1$: Da mayor peso a las observaciones recientes (responde rápidamente a cambios).
- $\alpha \approx 0$: Da mayor peso a las observaciones distantes (produce un suavizado más estable).

### Fórmulas Matemáticas

#### Forma Recurrente
$$\hat{Y}_{t+1|t} = \alpha Y_t + (1 - \alpha) \hat{Y}_{t|t-1}$$

Donde:
* $\hat{Y}_{t+1|t}$ es el pronóstico para el tiempo $t+1$ dado el conocimiento hasta el instante $t$.
* $Y_t$ es el valor observado en el tiempo $t$.
* $\alpha$ es la constante de suavizado ($0 \le \alpha \le 1$).

#### Forma de Nivel / Componente
$$L_t = \alpha Y_t + (1 - \alpha) L_{t-1}$$
$$\hat{Y}_{t+h|t} = L_t \quad \forall h \ge 1$$

#### Expandida en términos de observaciones pasadas
$$\hat{Y}_{t+1|t} = \alpha Y_t + \alpha(1-\alpha) Y_{t-1} + \alpha(1-\alpha)^2 Y_{t-2} + \dots + (1-\alpha)^t \hat{Y}_1$$

</div>

---

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

# Generar datos simulados
np.random.seed(42)
t = pd.date_range("2023-01-01", periods=60, freq="D")
y = 20 + np.random.normal(0, 2, size=60)
df_ses = pd.DataFrame({"Valor": y}, index=t)

# Ajuste del modelo SES con statsmodels
fit_alpha_02 = SimpleExpSmoothing(df_ses["Valor"]).fit(smoothing_level=0.2, optimized=False)
fit_alpha_08 = SimpleExpSmoothing(df_ses["Valor"]).fit(smoothing_level=0.8, optimized=False)
fit_auto = SimpleExpSmoothing(df_ses["Valor"]).fit(optimized=True)

df_ses["SES_alpha_0.2"] = fit_alpha_02.fittedvalues
df_ses["SES_alpha_0.8"] = fit_alpha_08.fittedvalues
df_ses["SES_Optimo"] = fit_auto.fittedvalues

# Gráfico interactivo con Plotly Graph Objects (go)
fig = go.Figure()

fig.add_trace(go.Scatter(x=df_ses.index, y=df_ses["Valor"], mode="markers+lines", name="Observado", marker=dict(size=5, color="black")))
fig.add_trace(go.Scatter(x=df_ses.index, y=df_ses["SES_alpha_0.2"], mode="lines", name="$\alpha = 0.2$", line=dict(color="#EF553B", dash="dash")))
fig.add_trace(go.Scatter(x=df_ses.index, y=df_ses["SES_alpha_0.8"], mode="lines", name="$\alpha = 0.8$", line=dict(color="#00CC96")))
fig.add_trace(go.Scatter(x=df_ses.index, y=df_ses["SES_Optimo"], mode="lines", name=f"$lpha$ Óptimo ({fit_auto.model.params['smoothing_level']:.2f})", line=dict(color="#636EFA", width=2.5)))

fig.update_layout(
    title="<b>Suavización Exponencial Simple (SES) con Diferentes $\alpha$</b>",
    xaxis_title="Fecha",
    yaxis_title="Nivel",
    template="plotly_white"
)
fig.show()

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

---

## 4. Método de Holt-Winters

### Concepto
El método de **Holt-Winters** (o Suavización Exponencial Triple) extiende SES para series de tiempo que presentan tanto **tendencia** como **estacionalidad**. Incorpora tres ecuaciones de suavización exponencial diferenciadas:
1. **Nivel ($L_t$):** Estimación del nivel actual de la serie.
2. **Tendencia ($b_t$ o $T_t$):** Estimación de la pendiente de la serie.
3. **Estacionalidad ($S_t$):** Estimación de los componentes estacionales con período $m$.

Existen dos variantes principales:
* **Holt-Winters Aditivo:** Se prefiere cuando las fluctuaciones estacionales son aproximadamente constantes a lo largo del tiempo.
* **Holt-Winters Multiplicativo:** Se prefiere cuando las fluctuaciones estacionales varían proporcionalmente al nivel de la serie.

### Fórmulas Matemáticas

#### 1. Holt-Winters Aditivo

* **Nivel:**
$$L_t = \alpha (Y_t - S_{t-m}) + (1 - \alpha) (L_{t-1} + b_{t-1})$$

* **Tendencia:**
$$b_t = \beta (L_t - L_{t-1}) + (1 - \beta) b_{t-1}$$

* **Estacionalidad:**
$$S_t = \gamma (Y_t - L_{t-1} - b_{t-1}) + (1 - \gamma) S_{t-m}$$

* **Ecuación de Pronóstico ($h$ pasos adelante):**
$$\hat{Y}_{t+h|t} = L_t + h b_t + S_{t-m + (h \pmod m)}$$

---

</div>

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

#### 2. Holt-Winters Multiplicativo

* **Nivel:**
$$L_t = \alpha \left(\frac{Y_t}{S_{t-m}} \right) + (1 - \alpha)(L_{t-1} + b_{t-1})$$

* **Tendencia:**
$$b_t = \beta (L_t - L_{t-1}) + (1 - \beta) b_{t-1}$$

* **Estacionalidad:**
$$S_t = \gamma \left(\frac{Y_t}{L_{t-1} + b_{t-1}} \right) + (1 - \gamma) S_{t-m}$$

* **Ecuación de Pronóstico ($h$ pasos adelante):**
$$\hat{Y}_{t+h|t} = (L_t + h b_t) 	imes S_{t-m + (h \pmod m)}$$

Donde:
- $\alpha \in [0,1]$: Parámetro de suavización del nivel.
- $\beta \in [0,1]$: Parámetro de suavización de la tendencia.
- $\gamma \in [0,1]$: Parámetro de suavización de la estacionalidad.
- $m$: Período estacional (ej. $m=12$ para datos mensuales, $m=4$ para trimestrales).

---

</div>

In [9]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Crear dataset simulado con tendencia y estacionalidad multiplicativa
np.random.seed(42)
periods = 48 # 4 años de datos mensuales
dates = pd.date_range("2020-01-01", periods=periods, freq="M")
t = np.arange(periods)
trend = 100 + 2.5 * t
seasonal = 1 + 0.2 * np.sin(2 * np.pi * t / 12)
noise = np.random.normal(0, 3, periods)
y = trend * seasonal + noise

df_hw = pd.DataFrame({"Valor": y}, index=dates)

# Ajuste del Modelo Holt-Winters Multiplicativo
model_hw = ExponentialSmoothing(
    df_hw["Valor"], 
    trend="add", 
    seasonal="mul", 
    seasonal_periods=12
).fit()

# Pronóstico a 12 meses futuros
forecast_steps = 12
future_dates = pd.date_range(start=dates[-1] + pd.DateOffset(months=1), periods=forecast_steps, freq="M")
forecast_values = model_hw.forecast(forecast_steps)

# Gráfico con Plotly Graph Objects (go)
fig = go.Figure()

# Datos ajustados e históricos
fig.add_trace(go.Scatter(x=df_hw.index, y=df_hw["Valor"], mode="lines+markers", name="Histórico Real", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=df_hw.index, y=model_hw.fittedvalues, mode="lines", name="Valores Ajustados", line=dict(color="#2ca02c", dash="dot")))

# Pronóstico
fig.add_trace(go.Scatter(x=future_dates, y=forecast_values, mode="lines+markers", name="Pronóstico (12 Meses)", line=dict(color="#d62728", width=3)))

fig.update_layout(
    title="<b>Pronóstico con el Método de Holt-Winters (Multiplicativo)</b>",
    xaxis_title="Fecha",
    yaxis_title="Ventas / Demanda",
    template="plotly_white"
)
fig.show()

/var/folders/_6/0ld42hn97t9_vc9j7kzt2xg80000gn/T/ipykernel_12688/3405208951.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range("2020-01-01", periods=periods, freq="M")
/var/folders/_6/0ld42hn97t9_vc9j7kzt2xg80000gn/T/ipykernel_12688/3405208951.py:28: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  future_dates = pd.date_range(start=dates[-1] + pd.DateOffset(months=1), periods=forecast_steps, freq="M")


<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

---

## 5. Introducción a los Modelos ARMA(p,q)

### Concepto
Los modelos **ARMA(p, q)** (Autoregressive Moving Average) combinan dos componentes fundamentales para describir y pronosticar series de tiempo **estacionarias** (cuya media, varianza y autocovarianza son independientes del tiempo):

1. **Componente Autorregresivo AR(p):** Describe la variable de interés como una combinación lineal de sus propios valores pasados (retardos o *lags*). $p$ indica el número de retardos incluidos.
2. **Componente de Media Móvil MA(q):** Modela la variable como una combinación lineal del ruido blanco actual y los errores de pronóstico del pasado. $q$ representa el número de retardos de los errores aleatorios.

* **Requisito de Estacionariedad:** Un proceso $ARMA(p,q)$ requiere que la serie subyacente sea estacionaria. Si presenta tendencia o no-estacionariedad en media, se requiere aplicar diferenciación (dando origen a los modelos $ARIMA(p,d,q)$).

---

### Fórmulas Matemáticas

#### 1. Proceso Autorregresivo AR(p)
$$Y_t = c + \sum_{i=1}^{p} \phi_i Y_{t-i} + \epsilon_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \dots + \phi_p Y_{t-p} + \epsilon_t$$

#### 2. Proceso de Media Móvil MA(q)
$$Y_t = c + \epsilon_t + \sum_{j=1}^{q} 	\theta_j \epsilon_{t-j} = c + \epsilon_t + 	\theta_1 \epsilon_{t-1} + 	\theta_2 \epsilon_{t-2} + \dots + 	\theta_q \epsilon_{t-q}$$

#### 3. Modelo Combinado ARMA(p, q)
$$Y_t = c + \sum_{i=1}^{p} \phi_i Y_{t-i} + \epsilon_t + \sum_{j=1}^{q} 	\theta_j \epsilon_{t-j}$$

Forma explícita completa:
$$Y_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \dots + \phi_p Y_{t-p} + \epsilon_t + 	\theta_1 \epsilon_{t-1} + 	\theta_2 \epsilon_{t-2} + \dots + 	\theta_q \epsilon_{t-q}$$

#### Operador de Retardo (Lag Operator $B$ o $L$)
Usando el operador de retardo definido como $B^k Y_t = Y_{t-k}$:
$$\Phi(B) Y_t = c + \Theta(B) \epsilon_t$$

Donde:
* $\Phi(B) = 1 - \phi_1 B - \phi_2 B^2 - \dots - \phi_p B^p$
* $\Theta(B) = 1 + 	\theta_1 B + 	\theta_2 B^2 + \dots + 	\theta_q B^q$
* $\epsilon_t \sim 	\text{WN}(0, \sigma^2)$ representa un proceso de ruido blanco con media cero y varianza constante $\sigma^2$.

---

### Diagnóstico e Identificación de Modelos

| Modelo | Función de Autocorrelación (ACF) | Función de Autocorrelación Parcial (PACF) |
| :--- | :--- | :--- |
| **AR(p)** | Decae exponencialmente o con ondas amortiguadas | Se interrumpe drásticamente después del retardo $p$ |
| **MA(q)** | Se interrumpe drásticamente después del retardo $q$ | Decae exponencialmente o con ondas amortiguadas |
| **ARMA(p, q)** | Decae exponencialmente o con ondas amortiguadas | Decae exponencialmente o con ondas amortiguadas |

---

</div>

In [10]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, pacf

# 1. Simulación de un Proceso ARMA(1, 1)
# ARMA Equation: Y_t = 0.7 * Y_{t-1} + e_t + 0.4 * e_{t-1}
np.random.seed(42)
ar_params = np.array([1, -0.7])  # Nota: en statsmodels el signo de AR se invierte
ma_params = np.array([1, 0.4])

arma_process = ArmaProcess(ar_params, ma_params)
simulated_data = arma_process.generate_sample(nsample=200)

df_arma = pd.DataFrame({"Simulado": simulated_data})

# 2. Ajuste del Modelo ARIMA(1,0,1)
model = ARIMA(df_arma["Simulado"], order=(1, 0, 1))
results = model.fit()

print(results.summary())

# 3. Cálculo de ACF y PACF para visualización
acf_vals = acf(df_arma["Simulado"], nlags=20)
pacf_vals = pacf(df_arma["Simulado"], nlags=20)
lags = np.arange(len(acf_vals))

# 4. Visualización con Plotly Graph Objects (go)
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Función de Autocorrelación (ACF)", "Función de Autocorrelación Parcial (PACF)")
)

# ACF Stem plot
fig.add_trace(go.Bar(x=lags, y=acf_vals, name="ACF", marker_color="#636EFA"), row=1, col=1)
# PACF Stem plot
fig.add_trace(go.Bar(x=lags, y=pacf_vals, name="PACF", marker_color="#EF553B"), row=1, col=2)

# Añadir bandas de significancia (95% confianza ≈ ±1.96 / sqrt(N))
conf_limit = 1.96 / np.sqrt(len(simulated_data))
for col in [1, 2]:
    fig.add_shape(type="line", x0=0, x1=20, y0=conf_limit, y1=conf_limit, line=dict(color="gray", dash="dash"), row=1, col=col)
    fig.add_shape(type="line", x0=0, x1=20, y0=-conf_limit, y1=-conf_limit, line=dict(color="gray", dash="dash"), row=1, col=col)

fig.update_layout(
    title="<b>Diagnóstico de Autocorrelación - Proceso ARMA(1, 1)</b>",
    template="plotly_white",
    showlegend=False
)
fig.show()

                               SARIMAX Results                                
Dep. Variable:               Simulado   No. Observations:                  200
Model:                 ARIMA(1, 0, 1)   Log Likelihood                -269.061
Date:                Mon, 27 Jul 2026   AIC                            546.123
Time:                        15:13:48   BIC                            559.316
Sample:                             0   HQIC                           551.462
                                - 200                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1763      0.260     -0.678      0.498      -0.686       0.334
ar.L1          0.6405      0.070      9.203      0.000       0.504       0.777
ma.L1          0.4141      0.083      4.968      0.0

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

## Resumen de Comparación de Modelos

| Método | Componentes Soportados | Sensibilidad a Tendencia | Sensibilidad a Estacionalidad | Complejidad / Uso Principal |
| :--- | :--- | :--- | :--- | :--- |
| **Descomposición Classica** | $T, S, C, I$ | Sí (filtrado) | Sí (índices fijos) | Análisis explicativo inicial |
| **Media Móvil** | Ninguno (solo suavizado) | Retraso (*lag*) | Sensible si $k 
eq m$ | Filtro simple de ruido |
| **SES** | Nivel | No | No | Series estables sin tendencia/estacionalidad |
| **Holt-Winters** | Nivel, Tendencia, Estacionalidad | Sí (aditiva/multiplicativa) | Sí (aditiva/multiplicativa) | Pronósticos a corto y mediano plazo |
| **ARMA(p, q)** | Dependencia de retardos y errores | No (requiere estacionariedad) | No (requiere SARIMA para estacionalidad) | Modelado estocástico riguroso |

</div>

---